# Merging NOAA, egalei, and ERA5

The goal of this notebook is to merge the NOAA (as a time series) and eaglei data using FIPS, datetime, and latitude/longitude (of the county centroid) as a multi-index

Then it will merge this with corresponding ERA5 data

## Imports

In [1]:
import pandas as pd
import dask.dataframe as dd
import xarray as xr
import fastparquet

# Indexing NOAA and eaglei by time

The eaglei data wasn't saved with a timeseries index, and didn't have FIPS as part of its index
This code will covert the original exported eaglei data into the right type of time series

In [2]:
#Load the ../Data/eaglei_data/eaglei_outages_with_county_info.parquet file
eaglei = pd.read_parquet('../Data/Merged_Data/eaglei_outages_with_county_info.parquet')

#Make datetime and FIPS a multiindex for eaglei
eaglei['time'] = pd.to_datetime(eaglei['datetime'])
eaglei.set_index(['time', 'FIPS'], inplace=True)

#Export eaglei as a parquet file
eaglei.to_parquet('../Data/Merged_Data/eaglei_outages_with_county_info_timeseries.parquet')

# Merging NOAA and eaglei

The datasets we want to work with are pretty large, so we'll load them as dask dataframes

In [3]:
#Load the ../Data/eaglei_data/eaglei_outages_with_county_info.parquet file
eaglei = dd.read_parquet('../Data/Merged_Data/eaglei_outages_with_county_info_timeseries.parquet')

#Load the ../Data/NOAA_Cleaned_ExplodedFIPS.parquet file
noaa = dd.read_parquet('../Data/NOAA_Cleaned_Data/NOAA_Timeseries.parquet')

#Merge eaglei and noaa based on their indices
eaglei_noaa = eaglei.merge(noaa, left_index=True, right_index=True, how='left')

#Export eaglei_noaa to a parquet file
eaglei_noaa.to_parquet('../Data/Merged_Data/eaglei_noaa.parquet', engine='pyarrow')

If we want to work with already-exported merged data, we can just load the file:

In [4]:
#Load the eaglei_noaa merged data
eaglei_noaa = pd.read_parquet('../Data/Merged_Data/eaglei_noaa.parquet')

# Adding Geospatial Indices to eaglei-NOAA

To merge with ERA5 data, the eaglei-NOAA data will need to have latitude and longitude as indices/coordinates.

In [ ]:
#Load the eaglei_noaa dataframe
eaglei_noaa = pd.read_parquet('../Data/Merged_Data/eaglei_noaa.parquet')

#Use fips_code, datetime, centroid_latitude and centroid_longitude as indices.
#Since dask doesn't support multiindexes, we'll keep these as variables as well
eaglei_noaa.set_index(['fips_code', 'datetime', 'centroid_latitude', 'centroid_longitude'], inplace=True, drop=False)

#Rename the indices FIPS, time, latitude, and longitude
eaglei_noaa.index.names = ['FIPS', 'time', 'latitude', 'longitude']

#Export eaglei_noaa as a parquet file eaglei_noaa_latlon
eaglei_noaa.to_parquet('../Data/Merged_Data/eaglei_noaa_latlon.parquet')

# Downloading ERA5 Data

We'll use the API from the Copernicus web store to download the following variables from the ERA5-Land Reanalysis data set:
- Temperature at 2M
- U and V components of wind at 10 M
- Snowfall
- Total Precipitation

Note that these data are sampled at every 9 km. One degree corresponds to approximately 111 km, so the data are sampled at roughly every tenth of a degree. However, this won't be exact, so we'll need to a bit of rounding to be able to merge the ERA5 data with the eaglei/NOAA data.

In [ ]:
import cdsapi
c = cdsapi.Client()
for year in range(2014, 2024):
    c.retrieve(
        'reanalysis-era5-land',
        {
            'product_type': 'reanalysis',
            'variable': [
                '10m_u_component_of_wind', '10m_v_component_of_wind', 
                '2m_temperature', 'snowfall', 'total_precipitation'
            ],
            'year': str(year),
            'month': [f'{month:02d}' for month in range(1, 13)],
            'day': [f'{day:02d}' for day in range(1, 32)],
            "time": ["00:00", "06:00", "12:00","18:00"],
            "format": "grib",
            "download_format": "zip",
            "area": [50, -125, 24, -66]
        },
        f'../Data/ERA5_Data/ERA5_{year}.grib')

# Converting ERA5 to parquet

To perform the merging of ERA5 and the NOAA-eaglei data, we'll need both to be in parquet (or, at least, non-grib) format

In [ ]:
# Start by converting the ERA5 grib files to parquet
# As part of this process, we'll also need to adjust the index to match the format of the eaglei_noaa index

grib_files = [
    '../Data/ERA5_Data/ERA5_2014.grib',
    '../Data/ERA5_Data/ERA5_2015.grib',
    '../Data/ERA5_Data/ERA5_2016.grib',
    '../Data/ERA5_Data/ERA5_2017.grib',
    '../Data/ERA5_Data/ERA5_2018.grib',
    '../Data/ERA5_Data/ERA5_2019.grib',
    '../Data/ERA5_Data/ERA5_2020.grib',
    '../Data/ERA5_Data/ERA5_2021.grib',
    '../Data/ERA5_Data/ERA5_2022.grib',
    '../Data/ERA5_Data/ERA5_2023.grib'
]

for grib_file in grib_files:
    # Load the GRIB file using the cfgrib engine
    grib_data = xr.open_dataset(grib_file, engine='cfgrib', decode_timedelta=True)

    # Convert to an xarray DataFrame
    df = grib_data.to_dataframe()

    # Reset the index to convert the multiindex to columns
    df.reset_index(inplace=True)

    # Remove the time, step, number, and surface variables
    df.drop(columns=['time', 'step', 'number', 'surface'], inplace=True)

    # Rename valid_time as time to facilitate merging with eaglei/NOAA
    df.rename(columns={'valid_time': 'time'}, inplace=True)

    # Round latitude and longitude to the nearest tenth of a degree
    # This is to ensure that the latitude and longitude match the format in the eaglei_noaa index
    df['latitude'] = df['latitude'].round(1)
    df['longitude'] = df['longitude'].round(1)

    # Make the index time, latitude, and longitude
    df.set_index(['time', 'latitude', 'longitude'], inplace=True)

    # Save as Parquet file with the same name but with .parquet extension
    output_file = grib_file.replace('.grib', '.parquet')
    df.to_parquet(output_file)

Ignoring index file '../Data/ERA5_Data/ERA5_2014.grib.5b7b6.idx' incompatible with GRIB file
Ignoring index file '../Data/ERA5_Data/ERA5_2018.grib.5b7b6.idx' older than GRIB file
Ignoring index file '../Data/ERA5_Data/ERA5_2023.grib.5b7b6.idx' older than GRIB file


# Merging NOAA-eaglei and ERA5

We can use pyarrow or dask to merge the parquet files

In [2]:
#Load the eaglei_noaa_latlon file
merged = pd.read_parquet('../Data/Merged_Data/eaglei_noaa_latlon.parquet')

parquet_files = [
    '../Data/ERA5_Data/ERA5_2014.parquet',
    '../Data/ERA5_Data/ERA5_2015.parquet',
    '../Data/ERA5_Data/ERA5_2016.parquet',
    '../Data/ERA5_Data/ERA5_2017.parquet',
    '../Data/ERA5_Data/ERA5_2018.parquet',
    '../Data/ERA5_Data/ERA5_2019.parquet',
    '../Data/ERA5_Data/ERA5_2020.parquet',
    '../Data/ERA5_Data/ERA5_2021.parquet',
    '../Data/ERA5_Data/ERA5_2022.parquet',
    '../Data/ERA5_Data/ERA5_2023.parquet'
]

for file in parquet_files:
    # Load the ERA data and merge it with the already-(partially)-merged data
    era = pd.read_parquet(file)
    merged = merged.merge(era, how="left", on=['time', 'latitude', 'longitude'])

    #In merged, for each pair of identical columns ending with _x and _y, combine them into a new column, taking the non-NaN value if it exists
    #This can be done by iterating over the columns and checking for pairs that end with _x and _y
    # Create a list to hold the new column names
    new_columns = []
    for col in merged.columns:
    # Check if the column ends with _x
        if col.endswith('_x'):
            # Create the corresponding column name by removing _x and adding _new
            new_col = col[:-2]  # Remove the last two characters (_x)
            new_columns.append(new_col)
            # Create the new column by taking the non-NaN values from both columns
            merged[new_col] = merged[col].combine_first(merged[col[:-2] + '_y'])
            # Drop the original columns to avoid duplicates
            merged.drop(columns=[col, col[:-2] + '_y'], inplace=True)
    # Remove the era data to save memory
    del era

#Save merged as a parquet file using fastparquet partitioning on fips_code
merged.to_parquet(
    '../Data/Merged_Data/eaglei_noaa_era5.parquet',
    engine='fastparquet',
    partition_cols=['fips_code']
)

#Save merged as a parquet file
merged.to_parquet(
    '../Data/Merged_Data/eaglei_noaa_era5_full.parquet',
    engine='fastparquet'
)